# 🏋️ Use Case 6 — Workout Efficacy Prediction
## Personal Notebook — Houssem

**Objective:** Predict workout efficacy category (`Burns_Calories_Bin`: Low / Medium / High)  
**Model:** Random Forest Classifier  
**Inputs:** BPM metrics, session duration, workout type, experience level, fat %, water intake

> Starts from `df_clean.csv` produced by `shared_data_prep.ipynb`.

---

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 60)

## 2. Load Clean Dataset

In [ ]:
df = pd.read_csv('df_clean.csv')
print(f'Shape: {df.shape}')
df.head(3)

## 3. Define Target & Features

`Burns_Calories_Bin` is a pre-engineered column already in the dataset (Low / Medium / High).  
We use physiological and workout columns as features.

In [ ]:
target = 'Burns_Calories_Bin'

# Check target distribution
print('Target distribution:')
print(df[target].value_counts())

# Plot it
df[target].value_counts().plot(kind='bar', color=['#4C9BE8','#F4A261','#2A9D8F'])
plt.title('Burns_Calories_Bin — Class Distribution')
plt.xlabel('Class')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Select relevant features for workout efficacy
feature_cols = [
    'Age', 'Gender', 'Weight (kg)', 'Height (m)',
    'Max_BPM', 'Avg_BPM', 'Resting_BPM',
    'Session_Duration (hours)', 'Workout_Type',
    'Fat_Percentage', 'Water_Intake (liters)',
    'Workout_Frequency (days/week)', 'Experience_Level', 'BMI'
]

df_uc6 = df[feature_cols + [target]].copy()
print(f'Working dataset shape: {df_uc6.shape}')

## 4. Preprocessing
### 4.1 Encode categorical columns

In [ ]:
# Encode Gender and Workout_Type (label encoding for tree-based models)
le = LabelEncoder()

for col in ['Gender', 'Workout_Type']:
    if col in df_uc6.columns:
        df_uc6[col] = le.fit_transform(df_uc6[col].astype(str))
        print(f'{col} encoded.')

# Encode target
df_uc6[target] = le.fit_transform(df_uc6[target].astype(str))
print(f'Target classes: {list(le.classes_)}')

### 4.2 Split features and target

In [ ]:
X = df_uc6.drop(target, axis=1)
y = df_uc6[target]

print('X shape:', X.shape)
print('y shape:', y.shape)
print('Missing values in X:', X.isnull().sum().sum())

### 4.3 Feature scaling

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

### 4.4 Train / Test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train size: {len(X_train)} | Test size: {len(X_test)}')

## 5. Model — Random Forest Classifier

In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
print('✅ Model trained.')

## 6. Evaluation

In [ ]:
y_pred = model.predict(X_test)

print('Accuracy:', round(accuracy_score(y_test, y_pred) * 100, 2), '%')
print()
print('Classification Report:')
print(classification_report(y_test, y_pred))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['High','Low','Medium'],
            yticklabels=['High','Low','Medium'])
plt.title('Confusion Matrix — Random Forest')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

## 7. Feature Importance

In [ ]:
importance = pd.Series(model.feature_importances_, index=X.columns)
importance = importance.sort_values(ascending=False)

plt.figure(figsize=(10, 5))
importance.plot(kind='bar', color='#4C9BE8')
plt.title('Feature Importance — Random Forest')
plt.ylabel('Importance')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(importance)

## 8. Conclusion

- **Model:** Random Forest Classifier
- **Target:** `Burns_Calories_Bin` (Low / Medium / High)
- **Key features:** Session duration, BPM metrics, workout type, experience level
- **Next steps:** Hyperparameter tuning with `GridSearchCV` if accuracy needs improvement